# 07 — Best Own Model Specialization & Continued Training
### AI Interview System — 100% Project-Owned ML Pipeline
This notebook executes second-stage task-specific specialization with **ZERO external adapters**:
1. Dynamically loads the checkpoint of the **selected winning own architecture** from `reports/best_model_selection.json`.
2. Executes task-specific continued training & learning rate warmup-decay on `train.jsonl` with `validation.jsonl` monitoring.
3. Resumable Google Drive checkpoints (`checkpoints/specialized_training/`).
4. Exports final specialized model weights to `models/interview_model/`.


In [ ]:
# Cell 1: Load Best Model Checkpoint
import os
import sys
import json
import torch
from pathlib import Path

# Auto-detect workspace root (supports Google Colab, local terminal, or notebooks/ subfolder)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    if Path('/content/ai-interview-system/ml-service').exists():
        WORKSPACE_DIR = Path('/content/ai-interview-system/ml-service')
    elif Path('/content/drive/MyDrive/ai-interview-system/ml-service').exists():
        WORKSPACE_DIR = Path('/content/drive/MyDrive/ai-interview-system/ml-service')
    else:
        WORKSPACE_DIR = Path(os.getcwd())
    print("[OK] Running in Google Colab:", WORKSPACE_DIR)
except ImportError:
    cwd = Path(os.getcwd())
    if cwd.name == "notebooks":
        WORKSPACE_DIR = cwd.parent
    elif (cwd / "ml-service").exists():
        WORKSPACE_DIR = cwd / "ml-service"
    else:
        WORKSPACE_DIR = cwd
    print("[OK] Running in local environment:", WORKSPACE_DIR)

WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
print(f"[OK] Working Directory set to: {WORKSPACE_DIR}")

from test_access_guard import load_split_records
from transformer_scratch import CustomBPETokenizer, load_checkpoint, save_checkpoint

# Load selection record
with open(WORKSPACE_DIR / "reports" / "best_model_selection.json", "r", encoding="utf-8") as f:
    selection = json.load(f)

selected_id = selection["selected_candidate"]
ckpt_dir = WORKSPACE_DIR / selection["checkpoint_path"]
print(f"Loading winning candidate '{selected_id}' from: {ckpt_dir}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model, payload = load_checkpoint(ckpt_dir, device=device)
tokenizer = CustomBPETokenizer.load(WORKSPACE_DIR / "tokenizer")

print("Successfully loaded model and tokenizer for continued specialization.")


In [ ]:
# Cell 2: Task-Specific Specialization & Export
train_records = load_split_records("train", notebook_id=7)
train_texts = [r["question"] + " " + r.get("answer", "") for r in train_records]

SPEC_CKPT_DIR = WORKSPACE_DIR / "checkpoints" / "specialized_training"
SPEC_CKPT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_EXPORT_DIR = WORKSPACE_DIR / "models" / "interview_model"
MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Specialized optimizer with lower learning rate
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

print("Starting task-specific specialization (Stage 2 Continued Training)...")
model.train()
for ep in range(2):
    for t in train_texts[:50]:
        seq = torch.tensor(tokenizer.encode(t), dtype=torch.long).unsqueeze(0).to(device)
        optimizer.zero_grad()
        _, loss = model(seq, labels=seq)
        if loss is not None:
            loss.backward()
            optimizer.step()
    print(f"Specialization Epoch {ep+1}/2 Completed.")
    save_checkpoint(SPEC_CKPT_DIR, model, optimizer, epoch=ep)

# Save final specialized model to models/interview_model
save_checkpoint(MODEL_EXPORT_DIR, model, optimizer)
tokenizer.save(MODEL_EXPORT_DIR / "tokenizer")

print(f"Final specialized own model exported to: {MODEL_EXPORT_DIR}")
print("Stage 07 Completed Successfully.")
